# Train LLaMA from Scratch — Google Colab

Train a model **from scratch** (random init, no pretrained weights) using:
- `config.py` — hyperparameters
- `data_loader.py` — streaming dataset
- `train.py` — training loop

**Colab T4 (16GB)**: Use 3B with `block_size=512`, gradient checkpointing, 8-bit optimizer.

## 1. Setup: Drive + GPU + pip

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
except Exception as e:
    DRIVE_MOUNTED = False
    print("Drive not mounted:", e)

!pip install -q torch transformers datasets accelerate bitsandbytes zstandard sentencepiece

# Disable safetensors auto-conversion (avoids JSONDecodeError in Colab)
import transformers.safetensors_conversion as _st_conv
_st_conv.auto_conversion = lambda *a, **k: None

# GPU required: Runtime → Change runtime type → GPU
!nvidia-smi

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00:00:0100:01
Sun Mar  1 13:05:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                       

## 2. Config (like config.py)

In [2]:
# --- Model: config only (weights random init) ---
model_name = "openlm-research/open_llama_3b_v2"  # Use config + tokenizer; weights from scratch
tokenizer_use_fast = False  # Required for OpenLLaMA

# --- Dataset (like config.py) ---
dataset_name = "mlfoundations/dclm-baseline-1.0"
streaming_buffer_size = 50_000  # Colab: smaller buffer
block_size = 512  # T4 16GB: 512; use 1024 if you have 24GB+

# --- Training ---
max_steps = 5000  # Colab: keep low; increase for longer runs
batch_size = 1
gradient_accumulation_steps = 8
learning_rate = 3e-4
warmup_steps = 200
logging_steps = 10
save_steps = 500
save_total_limit = 2

output_dir = "/content/drive/MyDrive/LLM_from_scratch" if DRIVE_MOUNTED else "./LLM_from_scratch"

use_bf16 = True
use_8bit_optimizer = True
max_grad_norm = 1.0
dataloader_num_workers = 0
report_to = "none"

print(f"Model config: {model_name} (from scratch)")
print(f"Block size: {block_size}, Output: {output_dir}")

Model config: openlm-research/open_llama_3b_v2 (from scratch)
Block size: 512, Output: /content/drive/MyDrive/LLM_from_scratch


## 3. Data loader (like data_loader.py)

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import IterableDataset

class StreamingDataset(IterableDataset):
    def __init__(self, dataset_name, tokenizer_name, split="train", buffer_size=10000, block_size=1024, tokenizer_use_fast=False):
        self.dataset = load_dataset(dataset_name, split=split, streaming=True)
        self.shuffled = self.dataset.shuffle(buffer_size=buffer_size)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=tokenizer_use_fast)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.block_size = block_size

    def __iter__(self):
        pad_token_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else self.tokenizer.eos_token_id
        for sample in self.shuffled:
            tokenized = self.tokenizer(
                sample['text'],
                truncation=True,
                max_length=self.block_size,
                padding="max_length",
                return_tensors=None,
            )
            input_ids = tokenized['input_ids']
            attention_mask = tokenized['attention_mask']
            labels = [tid if attention_mask[i] else -100 for i, tid in enumerate(input_ids)]
            yield {
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            }

print("StreamingDataset defined.")

StreamingDataset defined.


## 4. Load tokenizer and dataset

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=tokenizer_use_fast)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = StreamingDataset(
    dataset_name=dataset_name,
    tokenizer_name=model_name,
    buffer_size=streaming_buffer_size,
    block_size=block_size,
    tokenizer_use_fast=tokenizer_use_fast,
)

print("Tokenizer and streaming dataset ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/512k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27838 [00:00<?, ?it/s]

Tokenizer and streaming dataset ready.


## 5. Load model (from scratch) and train (like train.py)

In [5]:
import torch
from transformers import (
    LlamaConfig,
    LlamaForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

cuda_ok = torch.cuda.is_available()
bf16_ok = cuda_ok and torch.cuda.is_bf16_supported()
use_bf16_actual = use_bf16 and bf16_ok
if not cuda_ok:
    raise RuntimeError("No GPU. Set Runtime → Change runtime type → GPU.")
if use_bf16 and not bf16_ok:
    print("WARNING: bf16 not supported. Using fp16.")
print(f"GPU: {torch.cuda.get_device_name(0)}, bf16: {use_bf16_actual}")

torch.cuda.empty_cache()

# FROM SCRATCH: config only, random weights (no pretrained)
config = LlamaConfig.from_pretrained(model_name)
dtype = torch.bfloat16 if use_bf16_actual else torch.float16
model = LlamaForCausalLM(config=config).to(dtype)
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_steps=max_steps,
    learning_rate=learning_rate,
    warmup_steps=warmup_steps,
    logging_steps=logging_steps,
    save_steps=save_steps,
    save_total_limit=save_total_limit,
    bf16=use_bf16_actual,
    fp16=not use_bf16_actual,
    remove_unused_columns=False,
    report_to=report_to,
    gradient_checkpointing=True,
    dataloader_num_workers=dataloader_num_workers,
    max_grad_norm=max_grad_norm,
    optim="adamw_bnb_8bit" if use_8bit_optimizer else "adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Done. Saved to", output_dir)

GPU: Tesla T4, bf16: True


: 

: 

: 